# 02 — VAR-GFEVD Statistical Spillover Baseline
1. **Econometric Framework:** Implementation of the **Diebold-Yilmaz (2012)** Spillover Index using an order-invariant **Generalized Forecast Error Variance Decomposition (GFEVD)** based on Pesaran & Shin (1998).
2. **Total Spillover Index (TSI):** Quantifying what percentage of systemic return variance across the frozen 20-coin crypto market is driven by cross-asset contagion.
3. **Net Directional Connectedness Ranking:** Identifying which assets are net systemic transmitters of volatility vs. net shock receivers.
4. **Econometric Baseline Export:** Exporting [`data/processed/statistical_connectedness.csv`](../data/processed/statistical_connectedness.csv) to benchmark against `TempME` graph attention in Notebook 03.

In [1]:
%load_ext autoreload
%autoreload 2

import os
import sys
from pathlib import Path

# Add src directory to path
src_path = str(Path("../src").resolve())
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import torch
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

pio.templates.default = "plotly_dark"

from cryptograph.data.dataloader import load_processed_dataset
from cryptograph.models.baselines.var_spillover import VARSpilloverBaseline


c:\Users\daksh\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load Processed Dataset & Extract Common Active Period

We load `data/processed/crypto_graph_dataset.pt` and isolate the period where all 20 cryptocurrencies are simultaneously active (2,131 trading days). A balanced panel is required to compute stationary VAR coefficients without missing-data imputation artifacts.


In [2]:
dataset_path = Path("../data/processed/crypto_graph_dataset.pt")
data = load_processed_dataset(dataset_path)

features = data["features"]
active_mask = data["active_mask"]
asset_names = data["asset_names"]
timestamps = pd.to_datetime(data["timestamps"])

# Extract daily log returns
log_returns_df = pd.DataFrame(
    features[:, :, 0].numpy(),
    index=timestamps,
    columns=asset_names
)

# Common active mask
all_active_mask = active_mask.numpy().all(axis=1)
ret_active = log_returns_df.loc[all_active_mask]

print(f"Total Assets:         {len(asset_names)} cryptocurrencies")
print(f"Common Active Window: {ret_active.index[0].strftime('%Y-%m-%d')} to {ret_active.index[-1].strftime('%Y-%m-%d')}")
print(f"Sample Length:        {len(ret_active)} consecutive trading days")


Total Assets:         20 cryptocurrencies
Common Active Window: 2020-10-14 to 2026-09-24
Sample Length:        2131 consecutive trading days


## 2. Pairwise Correlation Heatmap & Inter-Asset Dependence

Before modeling directional spillover transmission, we inspect the static pairwise correlation matrix across all 20 cryptocurrencies.


In [3]:
corr_matrix = ret_active.corr()

fig_corr = px.imshow(
    corr_matrix,
    x=asset_names,
    y=asset_names,
    color_continuous_scale="Viridis",
    zmin=0.0,
    zmax=1.0,
    title="Pairwise Return Correlation Matrix (20 Cryptocurrencies)",
    labels=dict(color="Correlation")
)
fig_corr.update_layout(width=720, height=680)
fig_corr.show()

off_diag_vals = corr_matrix.values[~np.eye(len(corr_matrix), dtype=bool)]
print(f"Mean Inter-Asset Correlation: {off_diag_vals.mean():.3f} (Min: {off_diag_vals.min():.3f}, Max: {off_diag_vals.max():.3f})")


Mean Inter-Asset Correlation: 0.560 (Min: 0.206, Max: 0.819)


## 3. Fit Diebold-Yilmaz VAR Model & Variance Decomposition (GFEVD)

We fit a Vector Autoregressive model $\text{VAR}(p=2)$ on the daily log return series:
$$r_t = c + \sum_{l=1}^p \Phi_l r_{t-l} + \epsilon_t, \quad \epsilon_t \sim \mathcal{N}(0, \Sigma)$$

We then compute the **Generalized Forecast Error Variance Decomposition (GFEVD)** over a forecast horizon $H = 10$ days:
$$\theta_{ij}(H) = \frac{\sigma_{jj}^{-1} \sum_{h=0}^{H-1} (e_i' A_h \Sigma e_j)^2}{\sum_{h=0}^{H-1} (e_i' A_h \Sigma A_h' e_i)}$$

Where $\theta_{ij}(H)$ measures the proportion of asset $i$'s future return uncertainty explained by shocks originating from asset $j$.


In [4]:
baseline = VARSpilloverBaseline(maxlags=2, forecast_horizon=10)
baseline.fit(ret_active)

spillover_results = baseline.compute_spillover_table()
tsi = spillover_results["total_spillover_index"]
directional_table = spillover_results["directional_table"]
net_spillover = spillover_results["net_spillover"].sort_values(ascending=False)

print("════════════════════════════════════════════════════════════════════")
print(f"DIEBOLD-YILMAZ TOTAL SPILLOVER INDEX (TSI): {tsi:.2f}%")
print("════════════════════════════════════════════════════════════════════")
print(f"Interpretation: {tsi:.1f}% of total return forecast variance in the crypto market")
print("is attributable to cross-asset contagion, with the remainder due to idiosyncratic shocks.")


[2026-09-30 09:46:49] [INFO] [cryptograph.models.baselines.var_spillover] - Fitted VAR baseline with lag order: 2
════════════════════════════════════════════════════════════════════
DIEBOLD-YILMAZ TOTAL SPILLOVER INDEX (TSI): 85.44%
════════════════════════════════════════════════════════════════════
Interpretation: 85.4% of total return forecast variance in the crypto market
is attributable to cross-asset contagion, with the remainder due to idiosyncratic shocks.


c:\Users\daksh\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)


## 4. Directional Spillover Table (From Others vs. To Others)

The full Diebold-Yilmaz table summarizes the directional shock transmission budget across all market participants.


In [5]:
display(directional_table.round(2))


,BTC,ETH,BNB,SOL,XRP,DOGE,ADA,AVAX,LINK,DOT,...,BCH,LTC,NEAR,XLM,ETC,ATOM,XMR,FIL,SHIB,FROM
BTC,11.54,7.73,5.12,4.49,4.43,3.37,5.39,4.62,5.76,5.61,...,5.46,5.92,4.04,4.46,5.17,4.28,3.71,4.11,1.67,88.46
ETH,7.06,10.58,4.90,4.99,4.34,2.91,5.52,4.78,6.55,6.12,...,5.00,5.89,4.32,4.31,6.21,4.71,3.24,4.08,1.59,89.42
BNB,6.09,6.38,13.90,5.10,4.16,2.13,4.86,5.06,5.60,5.78,...,4.59,5.42,3.98,4.05,4.94,4.68,3.93,4.24,1.24,86.10
SOL,5.56,6.74,5.26,14.36,4.21,2.50,5.69,5.79,5.92,5.89,...,4.02,4.69,5.00,3.83,4.80,4.88,2.83,4.03,1.38,85.64
XRP,5.14,5.55,4.08,4.04,13.65,2.64,6.14,4.55,5.81,5.48,...,4.46,5.77,3.69,8.29,4.99,4.39,2.85,3.67,1.24,86.35
DOGE,5.92,5.60,3.12,3.57,3.68,20.12,5.04,4.16,5.04,4.98,...,4.93,5.23,3.57,4.28,5.98,3.97,2.67,3.72,1.97,79.88
ADA,5.46,6.09,4.08,4.66,5.25,2.90,11.63,5.74,6.23,6.60,...,4.32,5.36,4.45,5.81,5.18,5.21,2.67,4.39,1.27,88.37
AVAX,5.22,5.86,4.76,5.26,4.32,2.66,6.41,12.97,6.00,6.46,...,4.15,4.97,5.21,4.36,4.89,5.77,2.73,4.24,1.27,87.03
LINK,5.45,6.79,4.43,4.55,4.64,2.73,5.85,5.09,11.03,6.51,...,4.90,5.96,4.66,4.72,5.56,5.48,3.05,4.34,1.29,88.97
DOT,5.24,6.22,4.50,4.39,4.35,2.66,6.08,5.33,6.35,10.72,...,4.42,5.91,4.92,5.05,5.41,6.01,2.89,5.04,1.33,89.28


## 5. Net Systemic Connectedness Ranking (Transmitters vs. Receivers)

The **Net Spillover Index** is defined as:
$$\mathcal{S}_i = \text{Spillover To Others}_i - \text{Spillover From Others}_i$$
- $\mathcal{S}_i > 0$: **Net Transmitter** of systemic risk (e.g. Bitcoin, Ethereum).
- $\mathcal{S}_i < 0$: **Net Receiver** / consumer of external shocks (e.g. Solana, Avalanche).


In [6]:
net_df = pd.DataFrame({
    "Asset": net_spillover.index,
    "Net_Spillover": net_spillover.values,
    "Role": ["Net Transmitter" if v > 0 else "Net Receiver" for v in net_spillover.values]
})

fig_net = px.bar(
    net_df,
    x="Asset",
    y="Net_Spillover",
    color="Role",
    color_discrete_map={"Net Transmitter": "#00e676", "Net Receiver": "#ff1744"},
    title="Net Systemic Connectedness Ranking (Diebold-Yilmaz 2012 Baseline)",
    labels={"Net_Spillover": "Net Spillover Index (%)", "Asset": "Cryptocurrency"}
)
fig_net.update_layout(height=450, xaxis_tickangle=-45)
fig_net.show()

print("Top 3 Net Transmitters (Systemic Drivers):")
for a, v in net_spillover.head(3).items():
    print(f"{a:5s}: +{v:6.2f}%")

print("\nTop 3 Net Receivers (Vulnerable Victims):")
for a, v in net_spillover.tail(3).items():
    print(f"{a:5s}: {v:6.2f}%")


Top 3 Net Transmitters (Systemic Drivers):
ETH  : + 24.95%
DOT  : + 22.62%
LINK : + 19.88%

Top 3 Net Receivers (Vulnerable Victims):
TRX  : -22.93%
DOGE : -27.56%
SHIB : -40.98%


## 6. Export Econometric Connectedness Benchmark for Deep Learning Validation

We export the fitted net connectedness series to [`data/processed/statistical_connectedness.csv`](../data/processed/statistical_connectedness.csv). In **Notebook 03**, `ExplanationValidator` uses this file to compute the **Spearman Rank Correlation** against the node attention scores learned by `TempME` to evaluate alignment between econometric and deep learning explanations.


In [7]:
out_path = Path("../data/processed/statistical_connectedness.csv")
spillover_df = pd.DataFrame({
    "Asset": net_spillover.index,
    "Net_Spillover": net_spillover.values
})
spillover_df.to_csv(out_path, index=False)

print("════════════════════════════════════════════════════════════════════")
print(f"Ground-truth statistical connectedness exported to:")
print(f"   {out_path.resolve()}")
print(f"Verified {len(spillover_df)} asset rankings ready for Notebook 03!")
print("════════════════════════════════════════════════════════════════════")


════════════════════════════════════════════════════════════════════
Ground-truth statistical connectedness exported to:
   D:\GitHubProjects\CryptoGraph\data\processed\statistical_connectedness.csv
Verified 20 asset rankings ready for Notebook 03!
════════════════════════════════════════════════════════════════════


## 6. Export Econometric Connectedness Benchmark for Deep Learning Validation

We export the fitted net connectedness series to [`data/processed/statistical_connectedness.csv`](../data/processed/statistical_connectedness.csv). In **Notebook 03**, `ExplanationValidator` uses this file to compute the **Spearman Rank Correlation** against the node attention scores learned by `TempME` to evaluate alignment between econometric and deep learning explanations.
